In [ ]:
import pandas as pd
import numpy as np
import warnings
import shap
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sqlalchemy import create_engine
from dotenv import load_dotenv
import xgboost as xgb 
import traceback

warnings.filterwarnings('ignore')
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
df = pd.read_sql('SELECT * FROM aqi_data', con=engine)
print(f"Đã đọc {len(df):,} hàng từ MySQL")
SEASON_MAP = {0: 'Đông', 1: 'Xuân', 2: 'Hè', 3: 'Thu'}
df.columns = df.columns.str.strip().str.lower()
df['local_time'] = pd.to_datetime(df['local_time'], errors='coerce')
df['season_name'] = df['season'].map(SEASON_MAP)

print("⚙️ Đang tạo các đặc trưng dịch trễ (Lag) và trung bình trượt (Rolling)...")

# --- AQI LAGS ---
df['aqi_lag_1'] = df['aqi'].shift(1)
df['aqi_lag_2'] = df['aqi'].shift(2)
df['aqi_lag_3'] = df['aqi'].shift(3)
df['aqi_lag_6'] = df['aqi'].shift(6)
df['aqi_lag_12'] = df['aqi'].shift(12)
df['aqi_lag_24'] = df['aqi'].shift(24)
df['aqi_lag_48'] = df['aqi'].shift(48)
df['aqi_lag_168'] = df['aqi'].shift(168)

# --- WEATHER LAG 1 ---
weather_cols = ['clouds', 'precipitation', 'pressure', 'relative_humidity', 'temperature', 'uv_index', 'wind_speed']
for col in weather_cols:
    if col in df.columns:
        df[f'{col}_lag_1'] = df[col].shift(1)

# --- AQI ROLLING & EMA ---
df['aqi_roll_6'] = df['aqi'].shift(1).rolling(6).mean()
df['aqi_roll_12'] = df['aqi'].shift(1).rolling(12).mean()
df['aqi_roll_24'] = df['aqi'].shift(1).rolling(24).mean()
df['aqi_roll_48'] = df['aqi'].shift(1).rolling(48).mean()
df['aqi_ema_12'] = df['aqi'].shift(1).ewm(span=12).mean()
df['aqi_ema_24'] = df['aqi'].shift(1).ewm(span=24).mean()

# --- AQI TREND ---
df['aqi_trend_1'] = df['aqi_lag_1'] - df['aqi_lag_2']
df['aqi_trend_6'] = df['aqi_lag_1'] - df['aqi_lag_6']
df['aqi_trend_24'] = df['aqi_lag_1'] - df['aqi_lag_24']

# --- PM2.5 & PM10 FEATURES ---
df['pm25_lag_1'] = df['pm25'].shift(1)
df['pm25_lag_6'] = df['pm25'].shift(6)
df['pm25_lag_24'] = df['pm25'].shift(24)
df['pm25_roll_24'] = df['pm25'].shift(1).rolling(24).mean()
df['pm25_ema_24'] = df['pm25'].shift(1).ewm(span=24).mean()
df['pm10_lag_1'] = df['pm10'].shift(1)

FEATURES_MODEL = [
    'clouds_lag_1', 'precipitation_lag_1', 'pressure_lag_1',
    'relative_humidity_lag_1', 'temperature_lag_1', 'uv_index_lag_1', 'wind_speed_lag_1',
    'month', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'season',
    'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_3', 'aqi_lag_6', 'aqi_lag_12', 'aqi_lag_24', 'aqi_lag_48', 'aqi_lag_168',
    'aqi_roll_6', 'aqi_roll_12', 'aqi_roll_24', 'aqi_roll_48',
    'aqi_ema_12', 'aqi_ema_24',
    'aqi_trend_1', 'aqi_trend_6', 'aqi_trend_24',
    'pm25_lag_1', 'pm25_lag_6', 'pm25_lag_24', 'pm25_roll_24', 'pm25_ema_24',
    'pm10_lag_1'
]

TARGET_REG = 'aqi'
TARGET_CLS = 'aqi_label'  

if TARGET_CLS not in df.columns:
    df[TARGET_CLS] = pd.cut(df[TARGET_REG], bins=[-1, 100, 200, 999], labels=[0, 1, 2]).astype(int)

train = df[df['year'] < 2025].copy()
test = df[df['year'] == 2025].copy()

X_train = train[FEATURES_MODEL].bfill().ffill().astype(float)
X_test = test[FEATURES_MODEL].bfill().ffill().astype(float)

paths_to_try_reg = [
    "library_framework/best_model.pkl",
    "../library_framework/best_model.pkl",
    "best_model.pkl"
]

paths_to_try_cls = [
    "library_framework/xgboost_best_model.json",
    "../library_framework/xgboost_best_model.json",
    "xgboost_best_model.json"
]

xgb_reg = None
xgb_cls = None

for p in paths_to_try_reg:
    if os.path.exists(p):
        print(f"⏳ Tìm thấy file tại: '{p}'. Đang tải mô hình Regression...")
        xgb_reg = joblib.load(p)
        print("✅ Mô hình Regression đã được load thành công!")
        break

if xgb_reg is None:
    raise FileNotFoundError("❌ Không tìm thấy file 'best_model.pkl' ở bất kỳ đường dẫn mẫu nào!")

for p in paths_to_try_cls:
    if os.path.exists(p):
        print(f"⏳ Tìm thấy file tại: '{p}'. Đang tải mô hình Classification...")
        try:
            xgb_cls = xgb.XGBClassifier()
            xgb_cls.load_model(p)
            print("✅ Mô hình Classification đã được load từ file JSON thành công!")
        except Exception:
            xgb_cls = joblib.load(p)
            print("✅ Mô hình Classification đã được load bằng Joblib thành công!")
        break

if xgb_cls is None:
    raise FileNotFoundError("❌ Không tìm thấy file 'xgboost_best_model.json' ở bất kỳ đường dẫn mẫu nào!")

# ==============================================================================
# GIẢI THÍCH MÔ HÌNH VỚI SHAP TREEEXPLAINER 
# ==============================================================================
print("\n🧬 Đang chuẩn hóa ma trận dữ liệu 36 cột...")

X_train_shap = X_train[FEATURES_MODEL].bfill().ffill().astype(float)
print(f"📊 Kích thước ma trận Train: {X_train_shap.shape} (Đầy đủ 36 cột)")

X_shap = X_train_shap.sample(n=min(1000, len(X_train_shap)), random_state=42)

print("\n🧬 Khởi chạy tiến trình tính toán SHAP values...")
try:
    explainer_reg = shap.TreeExplainer(xgb_reg)
    shap_values_reg = explainer_reg(X_shap)
    print("✅ Tính xong SHAP Regression!")
    
    xgb_cls.feature_names = list(FEATURES_MODEL)
    xgb_cls.feature_types = None  
    
    X_shap_ndarray = X_shap.to_numpy()
    
    explainer_cls = shap.TreeExplainer(xgb_cls, data=X_shap_ndarray)
    shap_values_cls = explainer_cls(X_shap_ndarray)
    print("✅ Tính xong SHAP Classification!")

    # ==========================================================================
    # TRỰC QUAN HÓA VÀ XUẤT ĐỒ THỊ
    # ==========================================================================
    
    # Đồ thị 1: SHAP Summary Plot (Regression)
    print("📊 Hiển thị SHAP Summary Plot cho Mô hình Hồi quy...")
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values_reg, X_shap, show=False)
    plt.title("XGBoost Regression - SHAP Summary Plot", fontsize=14)
    plt.tight_layout()
    plt.savefig('charts/image_ead60c.png', dpi=300)
    plt.show()

    # Đồ thị 2: SHAP Summary Plot (Classification)
    print("📊 Hiển thị SHAP Summary Plot cho Mô hình Phân loại...")
    plt.figure(figsize=(10, 6))
    if hasattr(shap_values_cls, "values") and len(shap_values_cls.values.shape) == 3:
        shap.summary_plot(shap_values_cls.values[:, :, -1], X_shap, show=False)
    elif isinstance(shap_values_cls, list): 
        shap.summary_plot(shap_values_cls[-1], X_shap, show=False)
    else:
        shap.summary_plot(shap_values_cls, X_shap, show=False)
    plt.title("XGBoost Classification - SHAP Summary Plot", fontsize=14)
    plt.tight_layout()
    plt.savefig('charts/image_ead642.png', dpi=300)
    plt.show()

    # Đồ thị 3: Waterfall Plot (Giải thích ca bệnh ô nhiễm đỉnh điểm)
    if len(test) > 0 and TARGET_REG in test.columns:
        example_idx = (test[TARGET_REG] - 300).abs().idxmin()
        example_x = test.loc[[example_idx]][FEATURES_MODEL].bfill().ffill().astype(float)
        
        shap_single_reg = explainer_reg(example_x)
        
        print(f"\n💡 Trực quan Waterfall Plot cho thời điểm đỉnh điểm ô nhiễm")
        plt.figure(figsize=(11, 6))
        if len(shap_single_reg.shape) == 3:
            shap.plots.waterfall(shap_single_reg[0, :, -1], show=False)
        else:
            shap.plots.waterfall(shap_single_reg[0], show=False)
        plt.title(f"Waterfall Plot giải thích lý do AQI thực tế đạt {test.loc[example_idx, TARGET_REG]:.1f}", fontsize=12)
        plt.tight_layout()
        plt.savefig('charts/image_ead64a.png', dpi=300)
        plt.show()

    # Đồ thị 4: Dependency Plot (Tương tác PM2.5 x Độ ẩm)
    if 'pm25_lag_1' in FEATURES_MODEL and 'relative_humidity_lag_1' in FEATURES_MODEL:
        print("\n📈 Hiển thị Dependency Plot cho biến PM2.5 tương tác với Độ ẩm...")
        plt.figure(figsize=(10, 6))
        shap_vals_for_plot = shap_values_reg.values
        if len(shap_vals_for_plot.shape) == 3:
            shap_vals_for_plot = shap_vals_for_plot[:, :, -1]
            
        shap.dependence_plot(
            'pm25_lag_1', 
            shap_vals_for_plot, 
            X_shap, 
            interaction_index='relative_humidity_lag_1', 
            show=False
        )
        plt.title('Dependency Plot: Sự tương tác giữa PM2.5 và Độ ẩm lên kết quả AQI', fontsize=12)
        plt.tight_layout()
        plt.savefig('charts/image_ead666.png', dpi=300)
        plt.show()
        
    print("\n" + "═"*60)
    print("🏆 HOÀN THÀNH XUẤT SẮC: Toàn bộ biểu đồ SHAP đã vẽ và lưu thành công!")
    print("═"*60)

except Exception as e:
    print("❌ LỖI CHI TIẾT TẠI SHAP:")
    print(traceback.format_exc())

c:\Users\DELL\OneDrive - ptit.edu.vn\Documents\btl\Air-Quality-Index-in-Hanoi-\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️ Cảnh báo MySQL (Query thành công nhưng database không có dữ liệu (0 dòng).). Tự động chuyển sang đọc file CSV dự phòng...
✅ Đã load dữ liệu thành công từ file CSV cha.
⚙️ Đang tạo các đặc trưng dịch trễ (Lag) và trung bình trượt (Rolling)...
⏳ Tìm thấy file tại: '../library_framework/best_model.pkl'. Đang tải mô hình Regression...
✅ Mô hình Regression đã được load thành công!
⏳ Tìm thấy file tại: '../library_framework/xgboost_best_model.json'. Đang tải mô hình Classification...
✅ Mô hình Classification đã được load từ file JSON thành công!

🧬 Đang chuẩn hóa ma trận dữ liệu 36 cột...
📊 Kích thước ma trận Train: (25992, 36) (Đầy đủ 36 cột)

🧬 Khởi chạy tiến trình tính toán SHAP values...
✅ Tính xong SHAP Regression!


 36%|=======             | 2164/6000 [02:16<04:01]       